# CATS Dataset Audit

## Controlled Anomalies Time Series (CATS)

This notebook performs the initial audit of the publicly available CATS (Controlled Anomalies Time Series) Dataset.

The purpose of this phase is to understand and verify the structure, temporal properties, signals, labels, and anomaly metadata of the dataset before any feature engineering or anomaly-detection modelling begins.

### Audit Questions

1. What are the exact columns?
2. How many rows are there?
3. What is the timestamp range?
4. Is the timestamp actually 1-second apart?
5. Are there duplicate timestamps?
6. Are there missing timestamps?
7. Which columns are the 17 signals?
8. What are the data types?
9. What is the distribution of `y`?
10. What are the actual `category` values?
11. What does `metadata.csv` contain?
12. How many anomaly segments are there?
13. What are the actual anomaly categories?
14. Which channels are identified as root causes?
15. Which channels are affected?

### Important

This notebook is an **audit only**.

At this stage, we will not:
- train an anomaly-detection model;
- perform feature engineering;
- randomly split the time series;
- tune model parameters;
- modify the raw dataset;
- use ground-truth labels as model features.

The raw CATS data will be treated as read-only.

In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw CATS data
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "metadata.csv"

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Metadata file:", METADATA_PATH)

Project root: c:\Users\ADEGOKE\Desktop\CATS
Data file: c:\Users\ADEGOKE\Desktop\CATS\data\raw\cats\data.parquet
Metadata file: c:\Users\ADEGOKE\Desktop\CATS\data\raw\metadata.csv


## 1 Inspect the Parquet schema without loading the full dataset


In [8]:
import pyarrow.parquet as pq

# Read only the parquet schema
parquet_file = pq.ParquetFile(DATA_PATH)

print("Number of columns:", len(parquet_file.schema.names))
print("\nExact columns:")

for i, column in enumerate(parquet_file.schema.names, start=1):
    print(f"{i}.{column}")

Number of columns: 20

Exact columns:
1.aimp
2.amud
3.arnd
4.asin1
5.asin2
6.adbr
7.adfl
8.bed1
9.bed2
10.bfo1
11.bfo2
12.bso1
13.bso2
14.bso3
15.ced1
16.cfo1
17.cso1
18.y
19.category
20.__index_level_0__


### Question 1 — Conclusion

The CATS Parquet file contains 20 columns.

- 17 columns are system signals.
- `y` is the anomaly label.
- `category` identifies the anomaly category.
- `__index_level_0__` is an index column and is not a modelling feature.

The exact role and meaning of the 17 signal columns will be verified from the CATS documentation and metadata rather than inferred from their abbreviated names.

## 2. Number of Rows

In [9]:
# Get the exact number of rows from the Parquet metadata
num_rows = parquet_file.metadata.num_rows
print("Number of rows:", num_rows)

Number of rows: 5000000


### Question 2 — Conclusion

The CATS Parquet file contains exactly 5,000,000 rows.

This matches the official CATS Version 2 documentation, which states that the dataset contains 5 million timestamps.

## 3. Timestamp Range

In [12]:
# Inspect the Parquet metadata to see how the index is stored

metadata = parquet_file.metadata.metadata

print("Parquet metadata keys:")
for key in metadata:
    print(key)

Parquet metadata keys:
b'pandas'
b'ARROW:schema'


In [13]:
import json

pandas_metadata = metadata[b"pandas"]
pandas_metadata = json.loads(pandas_metadata.decode("utf-8"))

print(json.dumps(pandas_metadata, indent=2))

{
  "index_columns": [
    "__index_level_0__"
  ],
  "column_indexes": [
    {
      "name": null,
      "field_name": null,
      "pandas_type": "unicode",
      "numpy_type": "object",
      "metadata": {
        "encoding": "UTF-8"
      }
    }
  ],
  "columns": [
    {
      "name": "aimp",
      "field_name": "aimp",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "amud",
      "field_name": "amud",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "arnd",
      "field_name": "arnd",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "asin1",
      "field_name": "asin1",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "asin2",
      "field_name": "asin2",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "meta

In [14]:
# Inspect the timestamp index

df_index = pd.read_parquet(DATA_PATH).index

print("First timestamp:", df_index[0])
print("Last timestamp:", df_index[-1])
print("Data type:", df_index.dtype)

First timestamp: 2023-01-01 00:00:00
Last timestamp: 2023-02-27 20:53:19
Data type: datetime64[us]


### Question 3 — Conclusion

The CATS dataset covers the period from **2023-01-01 00:00:00** to **2023-02-27 20:53:19**.

The dataset contains 5,000,000 observations sampled at 1 Hz (one observation per second), which corresponds to approximately 57.87 days of continuous data.

The timestamp is stored as the DataFrame index with datetime type `datetime64[us]`.

## Question 4 — Is the timestamp actually 1 second apart?

We will check the differences between consecutive timestamps to verify the sampling interval.

In [16]:
# Check the time difference between consecutive timestamps

time_differences = df_index.to_series().diff().dropna()

print("Minimum difference:", time_differences.min())
print("Maximum difference:", time_differences.max())
print("Most common difference:", time_differences.mode().iloc[0])

Minimum difference: 0 days 00:00:01
Maximum difference: 0 days 00:00:01
Most common difference: 0 days 00:00:01


### Question 4 — Conclusion

The timestamps are consistently 1 second apart.

The minimum, maximum, and most common time difference between consecutive observations are all 1 second.

## Question 5 — Are there duplicate timestamps?

We will check whether any timestamp occurs more than once in the dataset.

In [17]:
duplicate_timestamps = df_index.duplicated().sum()
print("Number of duplicate timestamp:", duplicate_timestamps)

Number of duplicate timestamp: 0


### Question 5 — Conclusion

There are no duplicate timestamps in the CATS dataset.

Each timestamp identifies a unique observation.

## Question 6 — Are there missing timestamps?

We will check whether any expected 1-second timestamps are missing from the time series.

In [20]:
from pandas._libs import missing
# Check for missing timestamps

expected_timestamps = pd.date_range(
    start=df_index[0],
    end=df_index[-1],
    freq="1s"
)

missing_timestamps = expected_timestamps.difference(df_index)

print("Number of missing timestamps:", len(missing_timestamps))

Number of missing timestamps: 0


### Question 6 — Conclusion

There are no missing timestamps in the CATS dataset.

The timestamp sequence is continuous at 1-second intervals across the entire dataset.

## Question 7 — Which columns are the 17 signals?

We will identify the 17 system signal columns and separate them from the anomaly label and category fields.

In [21]:
# Identify the 17 signal columns

signal_columns = [
    column
    for column in parquet_file.schema.names
    if column not in ["y", "category", "__index_level_0__"]
]

print("Number of signal columns:", len(signal_columns))
print("\nSignal columns:")

for i, column in enumerate(signal_columns, start=1):
    print(f"{i}. {column}")

Number of signal columns: 17

Signal columns:
1. aimp
2. amud
3. arnd
4. asin1
5. asin2
6. adbr
7. adfl
8. bed1
9. bed2
10. bfo1
11. bfo2
12. bso1
13. bso2
14. bso3
15. ced1
16. cfo1
17. cso1


### Question 7 — Conclusion

The CATS dataset contains 17 system signal columns:

`aimp`, `amud`, `arnd`, `asin1`, `asin2`, `adbr`, `adfl`, `bed1`, `bed2`, `bfo1`, `bfo2`, `bso1`, `bso2`, `bso3`, `ced1`, `cfo1`, and `cso1`.

The meanings of these abbreviated signal names have not yet been inferred and will be verified from the official CATS documentation.

## Question 8 — What are the data types?

We will verify the data types of the 17 signal columns and the ground-truth fields.

In [22]:
# Inspect data types

data_type = pd.read_parquet(
    DATA_PATH,
    engine="pyarrow"
).dtypes

print(data_type)

aimp        float64
amud        float64
arnd        float64
asin1       float64
asin2       float64
adbr        float64
adfl        float64
bed1        float64
bed2        float64
bfo1        float64
bfo2        float64
bso1        float64
bso2        float64
bso3        float64
ced1        float64
cfo1        float64
cso1        float64
y           float64
category    float64
dtype: object


### Question 8 — Conclusion

All 17 system signal columns are stored as `float64`.

The `y` anomaly label and `category` field are also stored as `float64`.

The timestamp is stored separately as the DataFrame index with datetime type.

## Question 9 — What is the distribution of `y`?

We will examine the values of the anomaly label `y` and count how many observations belong to each label.

In [27]:
# Check the distribution of the anomaly label

y_distribution = pd.read_parquet(
    DATA_PATH,
    columns=["y"]
)["y"].value_counts().sort_index()

print(y_distribution)

y
0.0    4809900
1.0     190100
Name: count, dtype: int64


### Question 9 — Conclusion

The `y` field contains two observed values:

- `0.0`: 4,809,900 observations
- `1.0`: 190,100 observations

The dataset therefore contains substantially more normal observations than anomalous observations.

## Question 10 — What are the actual `category` values?

We will identify the distinct values present in the `category` field and count the observations belonging to each value.

In [28]:
# Check the distribution of anomaly categories

category_distribution = pd.read_parquet(
    DATA_PATH,
    columns=["category"]
)["category"].value_counts().sort_index()

print(category_distribution)

category
0.0     4809900
1.0       20400
2.0       12000
3.0       18000
4.0       15600
5.0        9900
6.0       12100
7.0       19100
8.0       32600
9.0       10400
10.0       6400
11.0       9100
12.0      19600
13.0       4900
Name: count, dtype: int64


### Question 10 — Conclusion

The `category` field contains values from `0.0` through `13.0`.

- `0.0` accounts for the normal observations.
- `1.0` through `13.0` represent the observed anomaly categories.

The meanings of the individual category values will be verified from the CATS documentation rather than inferred from the numeric labels.

## Question 11 — What does `metadata.csv` contain?

We will inspect the structure and columns of the anomaly metadata file without yet interpreting the individual anomaly categories.

In [29]:
# Inspect the anomaly metadata

metadata_df = pd.read_csv(METADATA_PATH)

print("Number of rows:", len(metadata_df))
print("\nColumns:")
for column in metadata_df.columns:
    print("-", column)

print("\nFirst 5 rows:")
print(metadata_df.head())


Number of rows: 200

Columns:
- start_time
- end_time
- root_cause
- affected
- category

First 5 rows:
            start_time             end_time root_cause  affected  category
0  2023-01-12 15:11:45  2023-01-12 15:20:05       bso3  ['cfo1']        12
1  2023-01-12 16:27:46  2023-01-12 17:51:06       bso3  ['cfo1']         1
2  2023-01-12 18:19:35  2023-01-12 18:36:15       bfo2  ['cso1']         8
3  2023-01-12 20:46:32  2023-01-12 20:51:32       bed2  ['ced1']         7
4  2023-01-13 05:57:10  2023-01-13 06:02:10       bfo1  ['cfo1']         9


### Question 11 — Conclusion

The `metadata.csv` file contains 200 anomaly-segment records and five fields:

- `start_time` — start of the anomaly segment.
- `end_time` — end of the anomaly segment.
- `root_cause` — identified root-cause signal.
- `affected` — signal(s) affected by the anomaly.
- `category` — numeric anomaly category.

The metadata provides segment-level ground truth that can later support event-level evaluation and analysis.

The meanings of the individual category values have not yet been determined and will be verified from the official CATS documentation.

## Question 12 — How many anomaly segments are there?

We will verify the number of anomaly segments recorded in `metadata.csv`.

In [30]:
number_of_segments = len(metadata_df)

print("Number of anomaly segments:", number_of_segments)

Number of anomaly segments: 200


### Question 12 — Conclusion

The metadata contains **200 anomaly segments**.

This confirms that the CATS dataset provides 200 documented anomaly events with corresponding segment-level metadata.

## Question 13 — What are the actual anomaly categories?

We will identify the distinct anomaly categories recorded in the anomaly metadata.

In [31]:
metadata_categories = (
    metadata_df["category"]
    .value_counts()
    .sort_index()
)

print(metadata_categories)

category
1     11
2     13
3     22
4     15
5     13
6     12
7     20
8     28
9     18
10    10
11     6
12    20
13    12
Name: count, dtype: int64


### Question 13 — Conclusion

The anomaly metadata contains **13 distinct anomaly categories**, numbered `1` through `13`.

All categories from `1` to `13` are represented among the 200 documented anomaly segments.

Category `0` is not present in the anomaly metadata because it represents normal observations in the main dataset.

The meanings of categories `1` through `13` will be verified from the official CATS documentation rather than inferred from their numeric values.

## Question 14 — Which channels are identified as root causes?

We will identify the distinct signal channels recorded as root causes in the anomaly metadata and count how often each occurs.

In [32]:
root_cause_distribution = (
    metadata_df["root_cause"]
    .value_counts()
    .sort_index()
)

print(root_cause_distribution)

root_cause
bed1    26
bed2    36
bfo1    24
bfo2    33
bso1    24
bso2    32
bso3    25
Name: count, dtype: int64


### Question 14 — Conclusion

The metadata identifies seven signal channels as root causes of the anomaly segments:

- `bed1`
- `bed2`
- `bfo1`
- `bfo2`
- `bso1`
- `bso2`
- `bso3`

These seven channels account for all 200 documented anomaly segments.

The `root_cause` field is ground-truth metadata and will not be used as a model feature.

## Question 15 — Which channels are affected?

We will identify the distinct signal channels recorded as affected by the documented anomaly segments.

In [ ]:
affected_channels = (
    metadata_df["affected"]
    .value_counts()
)

print(affected_channels)

affected
['cfo1']    73
['cso1']    65
['ced1']    62
Name: count, dtype: int64


: 

### Question 15 — Conclusion

The metadata identifies three signal channels as affected by the documented anomaly segments:

- `cfo1` — 73 segments
- `cso1` — 65 segments
- `ced1` — 62 segments

These affected channels are provided as ground-truth metadata and will not be used as model features.

This completes the 15-question initial dataset audit.

## Official CATS Documentation Findings

The official CATS Version 2 documentation confirms that the dataset contains 13 anomaly categories, numbered `1` through `13`.

The documentation does not define the individual meanings of categories `1` through `13`. Therefore, their meanings will not be inferred from the numeric category values.

The documentation does confirm that CATS contains deliberately injected anomaly segments, precise anomaly boundaries, root-cause channels, and affected channels for ground-truth evaluation.

# Curriculum Requirements and CATS Mapping

The CATS dataset will be evaluated against the requirements of Project.

The purpose of this mapping is to distinguish between capabilities provided by the dataset and capabilities that must be implemented or evaluated as part of this project.

| Curriculum requirement | Project status |
|---|---|
| Multivariate telemetry | Confirmed in CATS |
| Anomaly detection | Confirmed in CATS |
| Spikes / oscillations | Confirmed in CATS documentation |
| Gradual drift | Not established in CATS |
| Level shifts | Not established in CATS |
| Missing data | Not present in the original CATS data |
| Sensor freezes | Not established in CATS |
| Seasonal deviations | Not established in CATS |
| Rule-based baseline | To be implemented |
| Rolling statistical detector | To be implemented |
| Isolation Forest | To be implemented |
| Precision | To be evaluated |
| Recall | To be evaluated |
| False-alarm rate | To be evaluated |
| Detection delay | To be evaluated |
| Missing-data robustness | To be evaluated through controlled missing-data experiments |
| Behaviour under drift | To be evaluated |